# Fine-tune Sentence Transformer for Music Search

This notebook fine-tunes a pre-trained sentence transformer on your music-specific data.

**Training approach:** Contrastive learning with Multiple Negatives Ranking Loss
- Trains the model to rank positive (query, play) pairs higher than negative pairs
- Uses in-batch negatives for efficient training

**Expected results:**
- 10-30% improvement on music-specific queries
- Better understanding of genre combinations
- Improved location-based search
- Better semantic understanding of DJ comments

**Requirements:**
1. Upload `finetuning_train.jsonl` and `finetuning_val.jsonl`
2. Enable GPU (Runtime → Change runtime type → T4 GPU)
3. Training time: ~1-2 hours for 3 epochs

In [ ]:
# Install dependencies
!pip install -q sentence-transformers datasets accelerate

In [ ]:
import pandas as pd
import torch
from sentence_transformers import SentenceTransformer, InputExample, losses
from sentence_transformers.evaluation import EmbeddingSimilarityEvaluator
from torch.utils.data import DataLoader
import json

# Check GPU
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f"Using device: {device}")
if device == 'cuda':
    print(f"GPU: {torch.cuda.get_device_name(0)}")

In [ ]:
# Configuration
BASE_MODEL = 'sentence-transformers/all-mpnet-base-v2'  # Best quality base model
# BASE_MODEL = 'sentence-transformers/all-MiniLM-L12-v2'  # Faster alternative

TRAIN_FILE = 'finetuning_train.jsonl'
VAL_FILE = 'finetuning_val.jsonl'
OUTPUT_PATH = 'music-search-model'

# Training hyperparameters
EPOCHS = 3
BATCH_SIZE = 16  # Adjust based on GPU memory
LEARNING_RATE = 2e-5
WARMUP_STEPS = 100

print(f"Base model: {BASE_MODEL}")
print(f"Training for {EPOCHS} epochs")
print(f"Batch size: {BATCH_SIZE}")

In [ ]:
# Load training data
print("Loading training data...")

train_data = []
with open(TRAIN_FILE, 'r') as f:
    for line in f:
        item = json.loads(line)
        train_data.append(InputExample(texts=[item['query'], item['positive']]))

print(f"Loaded {len(train_data):,} training examples")

# Show sample
print("\nSample training example:")
print(f"  Query: {train_data[0].texts[0]}")
print(f"  Positive: {train_data[0].texts[1]}")

In [ ]:
# Load validation data
print("Loading validation data...")

val_queries = []
val_positives = []
val_scores = []  # All pairs get score 1.0 (positive pairs)

with open(VAL_FILE, 'r') as f:
    for line in f:
        item = json.loads(line)
        val_queries.append(item['query'])
        val_positives.append(item['positive'])
        val_scores.append(1.0)

print(f"Loaded {len(val_queries):,} validation examples")

In [ ]:
# Load pre-trained model
print(f"\nLoading base model: {BASE_MODEL}...")
model = SentenceTransformer(BASE_MODEL, device=device)
print(f"✓ Model loaded")
print(f"  Embedding dimension: {model.get_sentence_embedding_dimension()}")
print(f"  Max sequence length: {model.max_seq_length}")

In [ ]:
# Create data loader
train_dataloader = DataLoader(train_data, shuffle=True, batch_size=BATCH_SIZE)

# Use Multiple Negatives Ranking Loss
# This loss uses in-batch negatives for efficient contrastive learning
train_loss = losses.MultipleNegativesRankingLoss(model)

# Create evaluator
evaluator = EmbeddingSimilarityEvaluator(
    val_queries,
    val_positives,
    val_scores,
    name='music-search-eval'
)

print(f"\nTraining configuration:")
print(f"  Loss: Multiple Negatives Ranking Loss")
print(f"  Training steps: {len(train_dataloader) * EPOCHS:,}")
print(f"  Warmup steps: {WARMUP_STEPS}")
print(f"  Evaluation: After each epoch")

In [ ]:
# Fine-tune the model
print("\n" + "="*80)
print("STARTING FINE-TUNING")
print("="*80)

model.fit(
    train_objectives=[(train_dataloader, train_loss)],
    evaluator=evaluator,
    epochs=EPOCHS,
    warmup_steps=WARMUP_STEPS,
    output_path=OUTPUT_PATH,
    optimizer_params={'lr': LEARNING_RATE},
    evaluation_steps=len(train_dataloader),  # Evaluate after each epoch
    save_best_model=True,
    show_progress_bar=True
)

print("\n" + "="*80)
print("FINE-TUNING COMPLETE!")
print("="*80)

In [ ]:
# Load best model and test
print(f"\nLoading fine-tuned model from {OUTPUT_PATH}...")
finetuned_model = SentenceTransformer(OUTPUT_PATH, device=device)
print("✓ Fine-tuned model loaded")

# Test on sample queries
test_queries = [
    "psychedelic folk rock",
    "upbeat dance music",
    "indie rock from Portland",
    "ambient electronic",
    "funky groovy soul"
]

print("\nTesting on sample queries:")
print("="*80)

for query in test_queries:
    # Encode query
    query_emb = finetuned_model.encode(query, convert_to_numpy=True)
    
    # Find closest validation examples
    val_embs = finetuned_model.encode(val_positives[:100], convert_to_numpy=True)
    
    # Compute similarities
    from sklearn.metrics.pairwise import cosine_similarity
    sims = cosine_similarity([query_emb], val_embs)[0]
    
    # Get top 3
    top_indices = sims.argsort()[-3:][::-1]
    
    print(f"\nQuery: '{query}'")
    for i, idx in enumerate(top_indices, 1):
        print(f"  {i}. [{sims[idx]:.3f}] {val_positives[idx]}")

In [ ]:
# Save model for download
print(f"\n{'='*80}")
print("READY TO DOWNLOAD")
print(f"{'='*80}")

# Zip the model directory for easy download
!zip -r music-search-model.zip {OUTPUT_PATH}/

print(f"\n✓ Model saved to: {OUTPUT_PATH}/")
print(f"✓ Zipped for download: music-search-model.zip")
print(f"\nDownload music-search-model.zip and use it for generating embeddings!")
print(f"\nTo use the fine-tuned model:")
print(f"  1. Extract the zip file")
print(f"  2. In your embedding notebook, change MODEL_NAME to the extracted path")
print(f"  3. Generate embeddings with your fine-tuned model!")